# Human-in-the-Loop with `interrupt`

**What you will build:** a graph that **pauses mid-run** to wait for a person, then **resumes** exactly where it left off. In 0.5 we did this with `input()`, which only works in a live cell. LangGraph's `interrupt` does it properly: the run stops, its full state is checkpointed, and you can resume later — even from a web request. Maps to chapter 04d of n8n.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/24_human_in_the_loop.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — LangChain + LangGraph on OpenRouter. Run once.
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## A draft-approve-send graph

The agent drafts an email, then **stops** for approval before the irreversible send. `interrupt()` is what pauses it. Human-in-the-loop **requires a checkpointer** (2.3) — the state has to be saved in order to resume.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class State(TypedDict):
    request: str
    draft: str
    decision: str
    result: str

def write_draft(state: State) -> dict:
    r = llm.invoke([{"role": "system", "content": "Draft a short, professional email. Output only the body."},
                    {"role": "user", "content": state["request"]}])
    return {"draft": r.content}

def human_review(state: State) -> dict:
    # This call PAUSES the graph and hands the payload back to the caller.
    decision = interrupt({"draft": state["draft"], "question": "Approve this email? (yes/no)"})
    return {"decision": decision}          # <- filled with the resume value when we continue

def send(state: State) -> dict:
    if "yes" in state["decision"].lower():
        return {"result": "Email sent."}   # real send would go here
    return {"result": "Cancelled - nothing was sent."}

builder = StateGraph(State)
builder.add_node("write_draft", write_draft)
builder.add_node("human_review", human_review)
builder.add_node("send", send)
builder.add_edge(START, "write_draft")
builder.add_edge("write_draft", "human_review")
builder.add_edge("human_review", "send")
builder.add_edge("send", END)

graph = builder.compile(checkpointer=InMemorySaver())   # checkpointer is REQUIRED for interrupt

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("Mermaid image unavailable, showing text instead:\n")
    print(graph.get_graph().draw_mermaid())

## Run until the pause

Invoke it. The graph runs `write_draft`, then stops at `human_review`; the return value carries the interrupt payload.

In [ ]:
cfg = {"configurable": {"thread_id": "email-1"}}

paused = graph.invoke({"request": "Ask the vendor to move Tuesday's demo to Thursday."}, config=cfg)
print("PAUSED. The graph is waiting for a human.")
print("interrupt payload:", paused["__interrupt__"][0].value)

The graph is frozen at `human_review`, its state safely checkpointed under `thread_id='email-1'`. Nothing was sent. And "frozen" is inspectable — ask the checkpointer where the thread stands:

In [ ]:
snapshot = graph.get_state(cfg)
print("paused at node:", snapshot.next)          # ('human_review',)
print("draft in state:", snapshot.values["draft"][:80])

`snapshot.next` is how a real application finds its pending approvals: scan the threads, collect the ones whose `next` is the review node, and show them to a human — that's an approval inbox in a dozen lines (you'll wire it to HTTP in 3.0). Now the human decides, and we **resume** by invoking with a `Command(resume=...)` — the value becomes the return of `interrupt()`.

In [ ]:
# A person reviews the draft above and approves. Resume the SAME thread:
final = graph.invoke(Command(resume="yes"), config=cfg)
print("result:", final["result"])

The run picked up exactly where it stopped — `human_review` returned `"yes"`, and `send` ran. Because the state is checkpointed, the pause can last milliseconds or days, and resume from a different process entirely. That's real human-in-the-loop, and it's what the `input()` version in 0.5 could never do.

```{tip}
Put the interrupt right **before** the irreversible action, not around the whole graph — same principle as 0.5. Let the model draft freely; gate only the send.
```

And the path that makes the gate *real* — a second request, this time denied:

In [ ]:
cfg2 = {"configurable": {"thread_id": "email-2"}}
graph.invoke({"request": "Tell the whole company the demo is cancelled."}, config=cfg2)
final = graph.invoke(Command(resume="no"), config=cfg2)
print("result:", final["result"])

Approval flows are only as trustworthy as their "no" branch — test it as deliberately as the happy path (an approval gate that can't decline is theater).

```{warning}
One mechanic you must know before writing your own: on resume, **the interrupted node re-runs from its first line** — `interrupt()` doesn't freeze mid-function; the checkpoint is at the node boundary, and on resume the node executes again with the `interrupt()` call now returning the resume value. Here that's invisible because `human_review` does nothing before the `interrupt()`. But put an LLM call *above* an `interrupt()` in the same node and you'll pay for that call **twice** (and it may return something different the second time — 0.0). Rule: expensive or side-effecting work goes in its own node *before* the interrupting one — exactly why `write_draft` and `human_review` are separate here.
```

```{note}
You built human-in-the-loop from graph primitives so you understand it. In production, **LangChain 1.0 ships prebuilt middleware** — including human-in-the-loop, conversation summarization, and PII/content moderation — that wraps these patterns for you. Build it once by hand to learn it, then reach for middleware: the same philosophy as the rest of this course. And it isn't only LangGraph — **PydanticAI 2.x** also ships tool-approval flags for human-in-the-loop (you saw the `requires_approval` deferred-tool flow in 1.5) and durable-execution integrations (Temporal, DBOS), so you can pause-and-resume there too.
```

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add an edit option.** Let the human resume with `{"action": "edit", "text": "..."}` and have `send` use the edited text instead. **Done when:** the sent email is the human's version, not the model's.
2. **Build the inbox.** Start three requests on threads `email-3/4/5` without resuming, then write a loop that lists every thread whose `next` is `('human_review',)` along with its draft. **Done when:** you have a printed approval queue.
3. **Trip the re-execution trap on purpose.** Move the `llm.invoke` from `write_draft` *into* `human_review` (above the `interrupt`) and add a `print("drafting...")` next to it. Pause, resume, and count the prints. **Done when:** you've seen "drafting..." twice and can explain the warning above from evidence.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — Edit option:**

```python
def human_review(state: State) -> dict:
    decision = interrupt({"draft": state["draft"],
                          "question": "approve / reject / edit?"})
    if isinstance(decision, dict) and decision.get("action") == "edit":
        return {"decision": "yes", "draft": decision["text"]}     # human's text replaces the draft
    return {"decision": decision}

# resume with:  graph.invoke(Command(resume={"action": "edit", "text": "Short version: can we move to Thursday?"}), config=cfg)
```

The resume value can be any JSON-able payload — approval gates in real products are rarely yes/no, and `interrupt` doesn't force them to be.

**2 — The inbox:**

```python
pending = []
for tid in ["email-3", "email-4", "email-5"]:
    c = {"configurable": {"thread_id": tid}}
    snap = graph.get_state(c)
    if snap.next == ("human_review",):
        pending.append((tid, snap.values["draft"][:60]))

for tid, draft in pending:
    print(f"{tid}: {draft}...")
```

That list *is* the product: render it in a web page, wire each row's buttons to a `Command(resume=...)` call, and you've built the "Agent Inbox" pattern — 3.0 turns exactly this into HTTP endpoints.

**3 —** Two prints, two LLM calls, possibly two *different* drafts — the second one silently replacing the one the human actually reviewed. That last detail is why this isn't just a cost bug: **the human approved text that no longer exists.** Node boundaries are transaction boundaries; design them like it.
::::

**What's next:** in **2.5** we use LangGraph's native cycles for the **reflection** pattern — a generate/critique loop drawn as a real loop.